In [4]:
import os 
import sys
ROOT_DIR = os.path.dirname(os.path.dirname(__file__))
sys.path.append(ROOT_DIR)
from modules.loader import Loader
loader = Loader(r"C:\Users\kiran\Desktop\law ai\datasets\sc_data")
pdf_paths = loader.load_sc_judgment_pdfs()
print(f"Loaded {len(pdf_paths)} PDF files.")
print("Sample PDF path:", pdf_paths[0] if pdf_paths else "No PDFs found.")
print("Loader module is functioning correctly.")
print("All tests passed!")

NameError: name '__file__' is not defined

In [5]:
ROOT_DIR = os.path.dirname(os.path.dirname(__file__))


NameError: name '__file__' is not defined

In [ ]:
from modules.text_extractor import TextExtractor
extractor = TextExtractor()

for pdf_path in pdf_paths:
    text = extractor.extract_pdf(pdf_path)


In [ ]:
from modules.llm_manager import LLMManager
from dotenv import load_dotenv

load_dotenv()  # reads .env for GROQ_API_KEY

llm = LLMManager(
    provider="groq",
    model_name="llama-3.3-70b-versatile"
)


NameError: name 'text' is not defined

In [ ]:
from scripts.sc_judgements.metadata_builder import MetadataBuilder

builder = MetadataBuilder(llm)


In [ ]:
meta = builder.build_metadata(pdf_path, text)
print(meta)


In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
emb = model.encode("Hello legal world")
print(len(emb))


c:\Users\kiran\Desktop\law ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kiran\Desktop\law ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kiran\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate dev

384


In [5]:
import os
from chromadb import PersistentClient

DB_PATH = r"C:\Users\kiran\Desktop\law ai\vector_db\metadata"   # change if needed

client = PersistentClient(path=DB_PATH)

# Fetch collection
collection = client.get_or_create_collection("sc_metadata")

print("Collections:", client.list_collections())


Collections: [Collection(name=sc_metadata)]


In [6]:
data = collection.get()
print("Total items:", len(data["ids"]))


Total items: 13


In [7]:
for i in range(5):
    print(f"\nID: {data['ids'][i]}")
    print("Metadata:", data["metadatas"][i])
    print("Document Summary:", data["documents"][i][:150], "...")



ID: meta_0
Metadata: {'pdf_path': 'C:\\Users\\kiran\\Desktop\\law ai\\datasets\\sc_data\\supreme_court_judgments\\1950\\Abdulla_Ahmed_vs_Animendra_Kissen_Mitter_on_14_March_1950_1.PDF', 'case_name': 'Abdulla Ahmed vs Animendra Kissen Mitter on 14 March, 1950', 'year': '1950'}
Document Summary: CASE NAME: Abdulla Ahmed vs Animendra Kissen Mitter on 14 March, 1950
YEAR: 1950
CITATIONS: ['Equivalent citations: 1950 AIR 15, 1950 SCR 30, AIR 1950 ...

ID: meta_1
Metadata: {'case_name': 'Arjuna Lal Misra vs The State on 30 November, 1950', 'pdf_path': 'C:\\Users\\kiran\\Desktop\\law ai\\datasets\\sc_data\\supreme_court_judgments\\1950\\Arjuna_Lal_Misra_vs_The_State_on_30_November_1950_1.PDF', 'year': '1950'}
Document Summary: CASE NAME: Arjuna Lal Misra vs The State on 30 November, 1950
YEAR: 1950
CITATIONS: ['Equivalent citations: AIR1953SC411, AIR 1953 SUPREME COURT 411,  ...

ID: meta_2
Metadata: {'case_name': 'Ashutosh Lahiry vs The State Of Delhi And Anr. on 19 May, 1950', 'year': '195

In [9]:
import time

query_filter = {"year": "1950"}

start = time.time()
res = collection.get(where=query_filter)
end = time.time()

print("Results found:", len(res["ids"]))
print("Time taken:", end - start, "seconds")


Results found: 13
Time taken: 0.0017011165618896484 seconds


In [10]:
# fetch the first embedding
sample_embedding = collection.get(include=["embeddings"])["embeddings"][0]

query_result = collection.query(
    query_embeddings=[sample_embedding],
    n_results=3
)

print("Top matches:")
for id, meta in zip(query_result["ids"][0], query_result["metadatas"][0]):
    print(id, meta)


Top matches:
meta_0 {'case_name': 'Abdulla Ahmed vs Animendra Kissen Mitter on 14 March, 1950', 'pdf_path': 'C:\\Users\\kiran\\Desktop\\law ai\\datasets\\sc_data\\supreme_court_judgments\\1950\\Abdulla_Ahmed_vs_Animendra_Kissen_Mitter_on_14_March_1950_1.PDF', 'year': '1950'}
meta_4 {'case_name': 'A.M. Mair & Co vs Gordhandass Sagarmull on 30 November,', 'pdf_path': 'C:\\Users\\kiran\\Desktop\\law ai\\datasets\\sc_data\\supreme_court_judgments\\1950\\A_M_Mair_Co_vs_Gordhandass_Sagarmull_on_30_November_1950_1.PDF', 'year': '1950'}
meta_12 {'year': '1950', 'case_name': 'Commissioner Of Income-Tax, Bombay vs Ahmedbhai Umarbhai', 'pdf_path': 'C:\\Users\\kiran\\Desktop\\law ai\\datasets\\sc_data\\supreme_court_judgments\\1950\\Commissioner_Of_Income_Tax_Bombay_vs_Ahmedbhai_Umarbhai_Co_Bombay_on_4_May_1950_1.PDF'}


In [11]:
import time

start = time.time()

collection.query(
    query_embeddings=[sample_embedding],
    n_results=10
)

print("Search time:", time.time() - start)


Search time: 0.007315397262573242


In [12]:
from chromadb import PersistentClient
from langchain_huggingface import HuggingFaceEmbeddings

# DB_PATH = "vector_db/metadata"    # your metadata vectorstore path
QUERY = "1951 AIR"                # your search query

# Load vector DB
client = PersistentClient(path=DB_PATH)
collection = client.get_collection("sc_metadata")

# Load the same embedding model you used earlier
embedder = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

# Create embedding for query
query_emb = embedder.embed_query(QUERY)

# Search in Chroma
results = collection.query(
    query_embeddings=[query_emb],
    n_results=5
)

# Extract results
ids = results["ids"][0]
docs = results["documents"][0]
metas = results["metadatas"][0]

# Pretty print
print("\n=== Top Results for Query:", QUERY, "===\n")
for i in range(len(ids)):
    print(f"Result #{i+1}")
    print("ID:", ids[i])
    print("Title:", metas[i].get("case_name"))
    print("Year:", metas[i].get("year"))
    print("PDF Path:", metas[i].get("pdf_path"))
    print("Matched Content Sample:")
    print(docs[i][:200], "...\n")
    print("-" * 80)


c:\Users\kiran\Desktop\law ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Top Results for Query: 1951 AIR ===

Result #1
ID: meta_6
Title: Brij Bhushan And Another vs The State Of Delhi on 26 May, 1950
Year: 1950
PDF Path: C:\Users\kiran\Desktop\law ai\datasets\sc_data\supreme_court_judgments\1950\Brij_Bhushan_And_Another_vs_The_State_Of_Delhi_on_26_May_1950_1.PDF
Matched Content Sample:
CASE NAME: Brij Bhushan And Another vs The State Of Delhi on 26 May, 1950
YEAR: 1950
CITATIONS: ['Equivalent citations: 1950 AIR 129, 1950 SCR 605, AIR 1950 SUPREME', '1950 AIR  129            1950 SC ...

--------------------------------------------------------------------------------
Result #2
ID: meta_0
Title: Abdulla Ahmed vs Animendra Kissen Mitter on 14 March, 1950
Year: 1950
PDF Path: C:\Users\kiran\Desktop\law ai\datasets\sc_data\supreme_court_judgments\1950\Abdulla_Ahmed_vs_Animendra_Kissen_Mitter_on_14_March_1950_1.PDF
Matched Content Sample:
CASE NAME: Abdulla Ahmed vs Animendra Kissen Mitter on 14 March, 1950
YEAR: 1950
CITATIONS: ['Equivalent citations: 195